In [1]:
import sys

print(sys.executable)

/home/martin/OneDrive/Projects/Portfolio/Upwork-Market-Analysis/.venv/bin/python


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("A környezet működik.")


NumPy: 2.5.1
Pandas: 3.0.5
A környezet működik.


### Define Data Structure

Before collecting data, we define the structure of the dataset.
Each row represents one Upwork job posting.

In [3]:
columns = [
    "job_id",
    "category",
    "search_keyword",
    "title",
    "description",
    "mandatory_skills",
    "preferred_qualifications",
    "experience_level",
    "job_type",
    "project_type",
    "duration",
    "workload",
    "budget_type",
    "budget_min",
    "budget_max",
    "posted_date",
    "proposals",
    "job_url",
    "collected_at",
]

columns

['job_id',
 'category',
 'search_keyword',
 'title',
 'description',
 'mandatory_skills',
 'preferred_qualifications',
 'experience_level',
 'job_type',
 'project_type',
 'duration',
 'workload',
 'budget_type',
 'budget_min',
 'budget_max',
 'posted_date',
 'proposals',
 'job_url',
 'collected_at']

### Read Raw Job Files

In [4]:
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

#Functions to extract data from the text files

def extract_title(lines):
    for index, line in enumerate(lines):
        if line == "___":
            return lines[index + 1]

def extract_url(lines):
    for line in lines:
        if line.startswith("URL:"):
            return line.removeprefix("URL:")

def extract_posted_date(lines):
    for line in lines:
        if line.startswith("Posted"):
            return line.removeprefix("Posted ").strip()

def extract_experience_level(lines):
    experience_levels = {
        "Entry level",
        "Intermediate",
        "Expert"
    }

    for line in lines:
        if line in experience_levels:
            return line
        
def extract_job_type(lines):
    job_types = {
        "Hourly",
        "Fixed-price"
    }

    for line in lines:
        if line in job_types:
            return line

def extract_budget(lines):
    for index, line in enumerate(lines):

        if line.startswith("$"):
            if line.removeprefix("$").replace(",", "") is not float:
                continue
            first_amount = float(
                line.removeprefix("$").replace(",", "")
            )

            if (
                index + 4 < len(lines)
                and lines[index + 2] == "-"
                and lines[index + 4].startswith("$")
            ):
                second_amount = float(
                    lines[index + 4]
                    .removeprefix("$")
                    .replace(",", "")
                )

                return {
                    "budget_type": "range",
                    "budget_min": first_amount,
                    "budget_max": second_amount,
                }

            return {
                "budget_type": "single",
                "budget_min": first_amount,
                "budget_max": first_amount,
            }

    return {
        "budget_type": None,
        "budget_min": None,
        "budget_max": None,
    }

def extract_duration(lines):
    for index, line in enumerate(lines):
        if line.startswith("Duration"):
            return lines[index - 1]

    return None

def workload(lines):
    for index, line in enumerate(lines):
        if line.startswith("Duration"):
            return lines[index - 3]

    return None

def extract_description(lines):
    description_lines = []
    start_line = None

    for index, line in enumerate(lines):
        if line.startswith("Summary"):
            start_line = index + 1
            break

    if start_line is not None:
        for index, line in enumerate(lines[start_line:], start=start_line):
            if lines[index + 2] == "Hourly" or lines[index + 2] == "Fixed-price":
                break
            description_lines.append(line)

    return "\n".join(description_lines).strip()

def extract_category(lines):
    for line in lines:
        if line.startswith("category:"):
            new_line = line.removeprefix("category: ").strip()
            if new_line == "-":
                return None
            else:
                return new_line
    return None

def extract_search_keyword(lines):
    for line in lines:
        if line.startswith("search_keyword"):
            new_line = line.removeprefix("search_keyword: ").strip()
            if new_line == "-":
                return None
            else:
                return new_line

    return None

def extract_proposals(lines):
    for index, line in enumerate(lines):
        if line.startswith("Proposals:"):
            return lines[index + 1].strip()

    return None

def extract_project_type(lines):
    for line in lines:
        if line.startswith("Project Type:"):
            return line.removeprefix("Project Type:").strip()

    return None

def extract_mandatory_skills(lines):
    skills = []
    start_line = None

    for index, line in enumerate(lines):
        if line.startswith("Mandatory skills"):
            start_line = index + 1
            break

    if start_line is not None:
        for index, line in enumerate(lines[start_line:], start=start_line):
            if line == "Preferred Qualifications" or line == "Activity on this job":
                break
            skills.append(line)

    return "\n".join(skills).strip()

def extract_preferred_qualifications(lines):
    qualifications = []
    start_line = None

    for index, line in enumerate(lines):
        if line.startswith("Preferred Qualifications"):
            start_line = index + 1
            break

    if start_line is not None:
        for index, line in enumerate(lines[start_line:], start=start_line):
            if line == "Activity on this job":
                break
            qualifications.append(line)

    if start_line is None or qualifications == []:
        return None

    return "\n".join(qualifications).strip()

#data extraction from a files

folder = Path(PROJECT_ROOT / "data" / "raw" / "JobPageText")
output_file = Path(PROJECT_ROOT / "data" / "processed" / "jobs.csv")

jobs = []

start_job_id = 1

if output_file.exists():
    existing_jobs_df = pd.read_csv(output_file)
    start_job_id = existing_jobs_df["job_id"].max() + 1


for file_path in sorted(folder.glob("*.txt")):

    job_id = int(file_path.stem)

    if job_id < start_job_id:
        continue

    with file_path.open("r", encoding="utf-8") as file:
        lines = [line.strip() for line in file]

    collected_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    job_data = {
        "job_id": job_id,
        "category": extract_category(lines),
        "search_keyword": extract_search_keyword(lines),
        "title": extract_title(lines),
        "description": extract_description(lines),
        "mandatory_skills": extract_mandatory_skills(lines),
        "preferred_qualifications": extract_preferred_qualifications(lines),
        "experience_level": extract_experience_level(lines),
        "job_type": extract_job_type(lines),
        "project_type": extract_project_type(lines),
        "duration": extract_duration(lines),
        "workload": workload(lines),
        **extract_budget(lines),
        "posted_date": extract_posted_date(lines),
        "proposals": extract_proposals(lines),
        "job_url": extract_url(lines),
        "collected_at": collected_at,
    }

    jobs.append(job_data)

print(f"Collected data for {len(jobs)} jobs.")

#.csv file creation

if jobs:

    new_df = pd.DataFrame(jobs)

    if output_file.exists():
        existing_df = pd.read_csv(output_file)

        df = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        df = new_df

    df.to_csv(output_file, index=False, encoding="utf-8")



Collected data for 0 jobs.


In [5]:
title = "Revise a Systematic Review & Meta-analysis After Peer Review (Medical Research)"

df = pd.read_csv(output_file)

if title in df["title"].values:
    print("✔ Már szerepel az adatbázisban.")
else:
    print("❌ Még nincs benne.")

❌ Még nincs benne.


## Summary

In this notebook, a reproducible data collection pipeline was created to transform
raw Upwork job postings into a structured dataset suitable for analysis.

### Accomplishments

- Defined a standardized dataset structure.
- Parsed and cleaned manually collected job postings.
- Extracted relevant project metadata.
- Generated a structured CSV dataset for further analysis.

### Next Step

The cleaned dataset will be explored in the next notebook using exploratory
data analysis (EDA) techniques to identify market trends, common technologies,
and recurring project patterns.